### 파이썬을 이용한 팩트 체크 코드 구성

LLM(대규모 언어 모델)을 사용하여 제공된 텍스트의 팩트 체크를 파이썬 코드로 구현하려면, 크게 다음과 같은 단계를 거쳐야 합니다.

1. **텍스트 분석**: 팩트 체크할 문장을 여러 개의 사실(statement)로 분리합니다.
    
2. **질의 생성**: 각 사실에 대해 검증에 필요한 질문을 생성합니다.
    
3. **정보 검색**: 생성된 질문을 바탕으로 신뢰할 수 있는 출처(웹 검색, 데이터베이스 등)에서 정보를 검색합니다.
    
4. **검증**: 검색된 정보를 원래의 사실과 비교하여 사실 여부를 판단합니다.
    
5. **결과 보고**: 검증 결과를 종합하여 최종 보고서를 생성합니다.
    

이를 파이썬 코드로 구성하는 방법은 다음과 같습니다.

In [ ]:
import os
import argparse
import google.generativeai as genai
import openai
from dotenv import load_dotenv
import json

# .env 파일에서 환경 변수 로드
load_dotenv()

True

In [9]:
# Google Generative AI API 키 설정
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# OpenAI API 키가 설정되어 있는지 확인
if not os.getenv("OPENAI_API_KEY"):
    print(
        "경고: OpenAI API 키가 설정되지 않았습니다. gpt-4o-mini 모델을 사용할 수 없습니다."
    )

### OpenAI에서 제공하는 모델 목록 확인

In [ ]:
from openai import OpenAI

client = OpenAI()

for model in client.models.list():
    # gpt-4.1로 시작하는 모델과 gpt-5로 시작하는 모델만 출력
    if model.id.startswith("gpt-4.1") or model.id.startswith("gpt-5"):
        print(model.id)

gpt-5-nano
gpt-4.1-2025-04-14
gpt-4.1
gpt-4.1-mini-2025-04-14
gpt-4.1-mini
gpt-4.1-nano-2025-04-14
gpt-4.1-nano
gpt-5-chat-latest
gpt-5-2025-08-07
gpt-5
gpt-5-mini-2025-08-07
gpt-5-mini
gpt-5-nano-2025-08-07


In [15]:
# --- 모델 설정 ---
# 참고: gpt-4.1-mini는 아직 출시되지 않은 모델이므로, 현재 사용 가능한 유사 모델인 gpt-4o-mini로 대체합니다.
SUPPORTED_MODELS = {
    "gemini-pro": "gemini-2.5-pro",
    "gemini-flash": "gemini-2.5-flash",
    "gpt-mini": "gpt-5-mini",
}


def read_text_file(file_path):
    """지정된 경로의 텍스트 파일을 읽어 내용을 반환합니다."""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        print(f"오류: 파일을 찾을 수 없습니다 - {file_path}")
        return None


def get_llm_response(prompt, model_choice, is_json_output=False):
    """선택된 LLM 모델로부터 응답을 받아옵니다."""
    model_api_name = SUPPORTED_MODELS[model_choice]

    try:
        if model_choice.startswith("gemini"):
            model = genai.GenerativeModel(model_api_name)
            generation_config = genai.GenerationConfig(
                response_mime_type=(
                    "application/json" if is_json_output else "text/plain"
                )
            )
            response = model.generate_content(
                prompt, generation_config=generation_config
            )
            return response.text
        elif model_choice == "gpt-mini":
            if not OPENAI_API_KEY:
                raise ValueError("OpenAI API 키가 필요합니다.")
            client = openai.OpenAI(api_key=OPENAI_API_KEY)
            response_format = (
                {"type": "json_object"} if is_json_output else {"type": "text"}
            )
            completion = client.chat.completions.create(
                model=model_api_name,
                messages=[{"role": "user", "content": prompt}],
                response_format=response_format,
            )
            return completion.choices[0].message.content
    except Exception as e:
        return f"API 호출 중 오류 발생: {e}"


def fact_check_pipeline(text_content, model_choice):
    """요청된 5단계 팩트체크 파이프라인을 실행합니다."""

    print(f"\n===== '{SUPPORTED_MODELS[model_choice]}' 모델로 팩트 체크를 시작합니다. =====\n")

    # --- 1단계: 텍스트 분석 (핵심 사실 추출) ---
    print("--- 1단계: 텍스트 분석 및 핵심 주장 추출 ---")
    prompt_extract = f"""
    다음 텍스트에서 검증이 필요한 핵심적인 사실 주장(statement)들을 추출해줘.
    각 주장은 명확하고 간결한 단일 문장으로 만들어줘.
    결과는 JSON 형식의 리스트로 반환해줘. 예: {{"statements": ["주장 1", "주장 2"]}}

    원본 텍스트:
    {text_content}
    """
    statements_json_str = get_llm_response(prompt_extract, model_choice, is_json_output=True)
    try:
        statements = json.loads(statements_json_str)["statements"]
        print(f"✅ 총 {len(statements)}개의 핵심 주장을 추출했습니다.\n")
    except (json.JSONDecodeError, KeyError) as e:
        print(f"오류: 핵심 주장을 JSON 형식으로 파싱하는 데 실패했습니다. {e}")
        print("원본 응답:", statements_json_str)
        return

    final_report = []

    for i, statement in enumerate(statements):
        print(f"\n--- [{i+1}/{len(statements)}] 다음 주장에 대한 검증 시작 ---")
        print(f"📌 주장: {statement}\n")

        # --- 2단계: 질의 생성 ---
        print("--- 2단계: 검증을 위한 검색 질의 생성 ---")
        prompt_queries = f"""
        다음 주장의 사실 여부를 확인하기 위해 구글 검색에 사용할 효과적인 검색어 2-3개를 생성해줘.
        검색어는 핵심 키워드 위주로 간결하게 만들어줘.
        결과는 JSON 형식의 리스트로 반환해줘. 예: {{"queries": ["검색어 1", "검색어 2"]}}

        주장: "{statement}"
        """
        queries_json_str = get_llm_response(prompt_queries, model_choice, is_json_output=True)
        try:
            queries = json.loads(queries_json_str)["queries"]
            print(f"✅ 생성된 검색어: {queries}\n")
        except (json.JSONDecodeError, KeyError) as e:
            print(f"오류: 검색어를 JSON 형식으로 파싱하는 데 실패했습니다. {e}")
            print("원본 응답:", queries_json_str)
            continue

        # --- 3단계: 정보 검색 (Google Search Tool 사용) ---
        print("--- 3단계: 생성된 질의로 웹 정보 검색 ---")
        try:
            # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<< 이 부분이 수정되었습니다.
            search_tool = genai.GenerativeModel(
                model_name=SUPPORTED_MODELS[model_choice],
                tools=[genai.protos.Tool(
                    google_search_retrieval=genai.protos.GoogleSearchRetrieval()
                )]
            )
            # <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
            search_response = search_tool.generate_content(" ".join(queries))
            search_results = search_response.text
            print("✅ 웹 검색을 완료했습니다.\n")
        except Exception as e:
            print(f"오류: Google 검색 중 오류가 발생했습니다. {e}")
            search_results = "정보 검색에 실패했습니다."


        # --- 4단계: 검증 ---
        print("--- 4단계: 검색된 정보와 원본 주장 비교 및 검증 ---")
        prompt_verify = f"""
        당신은 전문 팩트 체커입니다. 아래 제공된 '검색 결과'를 바탕으로 '원본 주장'의 사실 여부를 판단해주세요.

        판단 기준:
        - 사실: 검색 결과가 원본 주장을 명백하게 뒷받침하는 경우.
        - 거짓: 검색 결과가 원본 주장을 명백하게 반박하는 경우.
        - 일부 사실: 주장의 일부는 맞지만, 부정확하거나 맥락을 벗어난 내용이 포함된 경우.
        - 확인 불가: 검색 결과만으로 사실 여부를 판단하기 어려운 경우.

        결과는 다음 JSON 형식에 맞춰서 작성해주세요.
        {{
            "statement": "원본 주장",
            "queries_used": ["사용한 검색어 목록"],
            "verification_result": "사실/거짓/일부 사실/확인 불가 중 하나",
            "explanation": "왜 그렇게 판단했는지에 대한 구체적이고 간결한 설명. 만약 '거짓'이나 '일부 사실'이라면 정확한 정보를 포함하여 설명해주세요."
        }}

        [원본 주장]:
        "{statement}"

        [사용한 검색어]:
        {queries}
        
        [검색 결과]:
        {search_results}
        """
        verification_json_str = get_llm_response(prompt_verify, model_choice, is_json_output=True)
        try:
            verification_result = json.loads(verification_json_str)
            print(f"✅ 검증 완료: {verification_result['verification_result']}\n")
            final_report.append(verification_result)
        except (json.JSONDecodeError, KeyError) as e:
            print(f"오류: 검증 결과를 JSON 형식으로 파싱하는 데 실패했습니다. {e}")
            print("원본 응답:", verification_json_str)
            final_report.append({
                "statement": statement,
                "queries_used": queries,
                "verification_result": "파싱 오류",
                "explanation": verification_json_str
            })

    # --- 5단계: 결과 보고 ---
    print("\n\n================= 📝 최종 팩트체크 보고서 =================\n")
    for report in final_report:
        print(f"📌 주장: {report['statement']}")
        print(f" V 결과: **{report['verification_result']}**")
        print(f" L 근거: {report['explanation']}")
        print(f"   (검색어: {report.get('queries_used', 'N/A')})")
        print("-" * 60)

In [16]:
def factChecker(file_path: str, model: str = "gemini-flash"):
    """
    파일 경로와 모델 이름을 인수로 받아 텍스트 파일의 팩트를 체크합니다.

    Args:
        file_path (str): 팩트체크할 텍스트 파일의 경로.
        model (str, optional): 사용할 LLM 모델 이름. 
                               기본값은 "gemini-flash"입니다.
                               SUPPORTED_MODELS 딕셔너리에 정의된 키 중 하나여야 합니다.
    """
    # 입력된 모델 이름이 지원되는 모델인지 확인 (선택 사항이지만 안정성을 높여줌)
    if model not in SUPPORTED_MODELS.keys():
        print(f"오류: 지원하지 않는 모델입니다: '{model}'")
        print(f"사용 가능한 모델: {', '.join(SUPPORTED_MODELS.keys())}")
        return

    # 파일 내용을 읽어옴
    text_content = read_text_file(file_path)
    
    # 파일 내용이 있을 경우에만 파이프라인 실행
    if text_content:
        fact_check_pipeline(text_content, model)

In [17]:
factChecker("test_transcript.txt")  # 기본 모델(gemini-flash) 사용
# factChecker("path/to/another_article.txt", model="gemini-pro") # 모델 지정


===== 'gemini-2.5-flash' 모델로 팩트 체크를 시작합니다. =====

--- 1단계: 텍스트 분석 및 핵심 주장 추출 ---
✅ 총 8개의 핵심 주장을 추출했습니다.


--- [1/8] 다음 주장에 대한 검증 시작 ---
📌 주장: 인공지능은 현재 빠르게 발전하고 있습니다.

--- 2단계: 검증을 위한 검색 질의 생성 ---
✅ 생성된 검색어: ['인공지능 발전 속도', 'AI 기술 발전 현황', '인공지능 급속 발전']

--- 3단계: 생성된 질의로 웹 정보 검색 ---
오류: Google 검색 중 오류가 발생했습니다. 400 Search Grounding is not supported.
--- 4단계: 검색된 정보와 원본 주장 비교 및 검증 ---
✅ 검증 완료: 확인 불가


--- [2/8] 다음 주장에 대한 검증 시작 ---
📌 주장: ChatGPT는 2023년에 출시되었습니다.

--- 2단계: 검증을 위한 검색 질의 생성 ---
✅ 생성된 검색어: ['ChatGPT 출시일', 'ChatGPT 언제 출시', 'ChatGPT 출시 연도']

--- 3단계: 생성된 질의로 웹 정보 검색 ---
오류: Google 검색 중 오류가 발생했습니다. 400 Search Grounding is not supported.
--- 4단계: 검색된 정보와 원본 주장 비교 및 검증 ---
✅ 검증 완료: 확인 불가


--- [3/8] 다음 주장에 대한 검증 시작 ---
📌 주장: ChatGPT 출시 후 대화형 AI에 대한 관심이 크게 증가했습니다.

--- 2단계: 검증을 위한 검색 질의 생성 ---
✅ 생성된 검색어: ['ChatGPT 출시 대화형 AI 관심 증가', 'ChatGPT 이후 대화형 AI 트렌드', '대화형 AI 시장 성장 ChatGPT 영향']

--- 3단계: 생성된 질의로 웹 정보 검색 ---
오류: Google 검색 중 오류가 발생했습니다. 400 Search Grounding is not supported.
--- 4